# Hyperparameter Tuning with Hyrax

This notebook will show some simple examples of hyperparameter tuning using Hyrax.

We'll make use of the HyraxCNN model and the CIFAR-10 dataset.
The hyperparameters we'll tune will include:
- Initial learning rates (0.01, 0.1)
- different learning rate schedulers
  - Default LR scheduler w/ gamma=1
  - Default LR scheduler w/ gamma=0.9
  - CosineAnnealing w/ T_Max=10

We'll be evaluating the performance of our model using validation loss at the end of the last epoch.

Many open source hyperparameter tuning libraries exist, in this notebook we'll demonstrate the use of the following:
- Optuna
- Ray Tune
- HyperOpt
- wandb (Weights and Bias)

## Setup common configurations

We'll use the same model and data request for each of the different examples.

In [1]:
model_name = "HyraxCNN"

data_request = {
    "train": {
        "data": {
            "dataset_class": "HyraxCifarDataset",
            "data_location": "./data",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
        },
    },
    "validate": {
        "data": {
            "dataset_class": "HyraxCifarDataset",
            "data_location": "./data",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
        },
    },
}

optimizer = "torch.optim.Adam"

## Optuna example

Optuna documentation - https://optuna.org/

In [ ]:
# % pip install optuna

In [ ]:
import optuna
from hyrax import Hyrax

h = Hyrax()
h.set_config("model.name", model_name)
h.set_config("data_request", data_request)
h.set_config("optimizer.name", optimizer)


def objective(trial):
    lr = trial.suggest_float("learning_rate", 0.01, 0.1)
    lr_scheduler = trial.suggest_categorical(
        "lr_scheduler",
        [
            ("torch.optim.lr_scheduler.ExponentialLR", {"gamma": 1.0}),
            ("torch.optim.lr_scheduler.ExponentialLR", {"gamma": 0.9}),
            ("torch.optim.lr_scheduler.CosineAnnealingLR", {"T_max": 10}),
        ],
    )

    h.set_config("'torch.optim.Adam'.lr", lr)
    h.set_config("scheduler.name", lr_scheduler[0])
    h.set_config(f"'{lr_scheduler[0]}'", lr_scheduler[1])

    model = h.train()

    return model.final_validation_metrics["loss"]


study = optuna.create_study()
study.optimize(objective, n_trials=6)

In [8]:
print(f"Best trial: {study.best_trial.number}")
print(f"Best parameters: {study.best_params}")
print(f"Best value: {study.best_value}")

print("All trials:")
for trial in study.trials:
    print(f" - Trial {trial.number}: Value: {trial.value}, Params: {trial.params}")

Best trial: 3
Best parameters: {'learning_rate': 0.023684100732004, 'lr_scheduler': ('torch.optim.lr_scheduler.ExponentialLR', {'gamma': 0.9})}
Best value: 1.5635895729064941
All trials:
 - Trial 0: Value: 2.302615165710449, Params: {'learning_rate': 0.03147371061606545, 'lr_scheduler': ('torch.optim.lr_scheduler.CosineAnnealingLR', {'T_max': 10})}
 - Trial 1: Value: 2.3012354373931885, Params: {'learning_rate': 0.04234299623942146, 'lr_scheduler': ('torch.optim.lr_scheduler.ExponentialLR', {'gamma': 0.9})}
 - Trial 2: Value: 2.306378126144409, Params: {'learning_rate': 0.03353047133712538, 'lr_scheduler': ('torch.optim.lr_scheduler.ExponentialLR', {'gamma': 1.0})}
 - Trial 3: Value: 1.5635895729064941, Params: {'learning_rate': 0.023684100732004, 'lr_scheduler': ('torch.optim.lr_scheduler.ExponentialLR', {'gamma': 0.9})}
 - Trial 4: Value: 2.302015542984009, Params: {'learning_rate': 0.0502613280276386, 'lr_scheduler': ('torch.optim.lr_scheduler.CosineAnnealingLR', {'T_max': 10})}
 - 

The loss curves for each of the examples.

![optuna_hparam_tuning_loss](../_static/screenshots/optuna_hparam_tuning_loss.png)

## Ray Tune example

Ray Tune documentation: https://docs.ray.io/en/latest/tune/index.html

In [4]:
# % pip install ray[tune]

In [ ]:
from ray import tune
from hyrax import Hyrax


def trainable(config):
    h = Hyrax()  # Must be inside — each trial runs in its own process
    h.set_config("model.name", model_name)
    h.set_config("data_request", data_request)
    h.set_config("optimizer.name", optimizer)

    h.set_config("'torch.optim.Adam'.lr", config["learning_rate"])
    h.set_config("scheduler.name", config["lr_scheduler"][0])
    h.set_config(f"'{config['lr_scheduler'][0]}'", config["lr_scheduler"][1])

    model = h.train()
    loss = model.final_validation_metrics["loss"]

    tune.report({"loss": loss})


search_space = {
    "learning_rate": tune.uniform(0.01, 0.1),
    "lr_scheduler": tune.choice(
        [
            ("torch.optim.lr_scheduler.ExponentialLR", {"gamma": 1.0}),
            ("torch.optim.lr_scheduler.ExponentialLR", {"gamma": 0.9}),
            ("torch.optim.lr_scheduler.CosineAnnealingLR", {"T_max": 10}),
        ]
    ),
}

tuner = tune.Tuner(
    trainable,
    param_space=search_space,
    tune_config=tune.TuneConfig(num_samples=6),
)
results = tuner.fit()

best = results.get_best_result(metric="loss", mode="min")
print(f"Best config: {best.config}")
print(f"Best loss: {best.metrics['loss']}")

## Hyperopt example

Hyperopt documentation: https://hyperopt.github.io/hyperopt/

In [5]:
# % pip install hyperopt

In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from hyrax import Hyrax

h = Hyrax()
h.set_config("model.name", model_name)
h.set_config("data_request", data_request)
h.set_config("optimizer.name", optimizer)


def objective(config):
    h.set_config("'torch.optim.Adam'.lr", config["learning_rate"])
    h.set_config("scheduler.name", config["lr_scheduler"][0])
    h.set_config(f"'{config['lr_scheduler'][0]}'", config["lr_scheduler"][1])

    model = h.train()
    loss = model.final_validation_metrics["loss"]

    return {"loss": loss, "status": STATUS_OK}


search_space = {
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.1),
    "lr_scheduler": hp.choice(
        "lr_scheduler",
        [
            ("torch.optim.lr_scheduler.ExponentialLR", {"gamma": 1.0}),
            ("torch.optim.lr_scheduler.ExponentialLR", {"gamma": 0.9}),
            ("torch.optim.lr_scheduler.CosineAnnealingLR", {"T_max": 10}),
        ],
    ),
}

trials = Trials()
best = fmin(objective, search_space, algo=tpe.suggest, max_evals=6, trials=trials)

best_params = space_eval(search_space, best)
print(f"Best parameters: {best_params}")
print(f"Best loss: {min(t['result']['loss'] for t in trials.trials)}")
print(f"All values: {[t['result']['loss'] for t in trials.trials]}")